# 第二天：Agent 智能体 Knowledge Base

这个 Notebook 演示 V3 如何召回多张 YAML 知识卡候选，再由 Agent 做语义判断并显示可追溯引用。

## 1-1. 导入统一接口

知识卡读取、匹配、Agent 调用和引用生成全部来自 `src/`。

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "teacher":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.facade import invoke
from src.retrieval import KNOWLEDGE_DIRECTORY, discover_knowledge_cards


## 1-2. 查看 YAML 知识卡目录

V3 每次调用都会递归扫描 `student/knowledge/` 中按学科和分类组织的所有 `.md` 文件。

Python 用标题、关键词和别名召回最多三张完整候选卡，Agent 再判断是否真的相关。

In [ ]:
cards = discover_knowledge_cards(KNOWLEDGE_DIRECTORY)
for card in cards:
    print(f"[{card.card_id}] {card.title} ({card.source})")
    print(f"  学科/分类/年级：{card.subject} / {card.category} / {card.grade}")
    print(f"  关键词：{'、'.join(card.keywords)}")
    print(f"  别名：{'、'.join(card.aliases)}")
    print(f"  核心规则：{card.core_rule}")
    print()


## 1-3. 显示回答、引用和检索过程

`citations` 保存 Agent 实际采用的原文证据，`trace` 区分候选召回、采用和未采用。

In [ ]:
def display_knowledge_result(result):
    if result["error"]:
        raise RuntimeError(result["error"])

    print(result["text"])
    for citation in result["citations"]:
        print(f"📚 引用 [{citation['id']}]：{citation['title']}")
        print(f"   来源：{citation['source']}")
        for match in citation["matches"]:
            method = match.get("method", "exact")
            print(f"   采用 {match['field']}（{method}）")
            print(f"   原文：{match['excerpt']}")
    for step in result["trace"]:
        candidate_ids = [item['id'] for item in step.get('candidates', [])]
        print(f"🔎 检索状态：{step['status']}，候选：{candidate_ids}")


## 1-4. 用同一道题比较 V2 和 V3

V2 没有知识卡依据，V3 会用别名 `times` 召回 `four times`，语义判断相关后生成稳定 ID 引用。

In [ ]:
question = "I ____ (see) this movie four times. 我应该先观察哪个线索？"
v2_result = invoke("V2", question)
v3_result = invoke("V3", question)

print("=== V2 ===")
print(v2_result["text"] if not v2_result["error"] else v2_result["error"])
print("\n=== V3 ===")
display_knowledge_result(v3_result)


## 1-5. 与 Knowledge Agent 连续对话

输入 `/exit` 可以结束对话。

对话中可修改现有卡片或新增 `.md` 卡片，下一轮调用会立即重新扫描。

错题：I _____ (see) this movie 4 times. 要用什么时态呢

In [ ]:
def run_v3_dialogue():
    history = []
    citations = []
    trace = []
    print("📚 Knowledge Agent 已就绪。")

    while True:
        student_message = input("学生：").strip()
        if student_message.lower() in {"/exit", "exit", "退出"}:
            print("期待下次交流。")
            return {"history": history, "citations": citations, "trace": trace}
        if not student_message:
            print("输入不能为空，请重新输入。")
            continue

        turn_result = invoke("V3", student_message, history=history)
        display_knowledge_result(turn_result)
        citations.extend(turn_result["citations"])
        trace.extend(turn_result["trace"])
        history.extend([
            {"role": "user", "content": student_message},
            {"role": "assistant", "content": turn_result["text"]},
        ])


In [ ]:
v3_session = run_v3_dialogue()


## 1-6. 修改知识卡后再次观察

尝试替换 YAML 中的关键词或别名，或者修改正文证据并保存，然后继续上一段对话。

如果没有召回候选，或候选经语义判断后不相关，结果会返回空引用并显示对应状态。